# VPS Maintenance

Routine maintenance keeps the server healthy, secure, and from running out of disk space at 3am.
This notebook covers: Docker cleanup, UFW audits, log management, service monitoring, and the security checklist.

## Docker Cleanup

Docker accumulates build cache, dangling images, stopped containers, and unused volumes. These pile up fast.

```bash
# See what Docker is using
docker system df

# Safe cleanup: removes only what is not in use
docker builder prune -f
docker image prune -f
docker container prune -f

# Nuclear option: removes everything not currently running
docker system prune -a -f
```

Run `docker system df` before and after to confirm space was freed.

**Do not** run `docker system prune -a` without checking `docker ps` first. It removes all stopped containers, which may include containers Coolify expects to restart.

## UFW Firewall Audit

After every deploy or service decommission, audit the firewall.

```bash
ufw status numbered
```

For each rule: is there still an active listener on that port?

```bash
# Check what is actually listening
ss -tlnp
```

### Removing a stale rule

```bash
# List rules with numbers
ufw status numbered

# Delete by number
ufw delete 3
```

**Rule:** `ufw status` must show only ports that have active listeners. Stale rules create a false security picture.

## Service Health Monitoring

### Check all running containers

```bash
docker ps --format 'table {{.Names}}\t{{.Status}}\t{{.Ports}}'
```

### Check systemd services

```bash
systemctl list-units --type=service --state=running
systemctl status coolify
```

### Quick health check script

```bash
#!/usr/bin/env bash
# health.sh - run this daily

echo "=== Docker ==="
docker ps --format 'table {{.Names}}\t{{.Status}}' | head -20

echo ""
echo "=== Disk ==="
df -h / | tail -1
docker system df

echo ""
echo "=== Memory ==="
free -h

echo ""
echo "=== Firewall ==="
ufw status
```

## Log Management

### View logs for a container

```bash
docker logs CONTAINER_NAME --tail 100 -f
```

### View journald (systemd)

```bash
journalctl -u coolify --since "1 hour ago"
journalctl -p err --since today
```

### Limit log size

Set Docker log limits in `/etc/docker/daemon.json`:

```json
{
  "log-driver": "json-file",
  "log-opts": {
    "max-size": "10m",
    "max-file": "3"
  }
}
```

Restart Docker: `systemctl restart docker`

## Disk Space

When disk fills up, services crash silently. Check before it becomes a problem.

```bash
# Overall usage
df -h /

# What is taking space
du -sh /var/lib/docker/*
du -sh /var/log/*

# Find big files
find / -xdev -type f -size +100M 2>/dev/null | sort -n
```

### Rotation: if space is critically low

```bash
# 1. Clean Docker first (usually the biggest win)
docker system prune -f

# 2. Rotate logs
journalctl --vacuum-time=7d

# 3. Clean apt cache
apt autoremove -y && apt clean
```

## The Fix-the-Config Rule

When you fix something on a running system, the fix is **not done** until:

1. The config file (systemd unit, `.env`, `compose.yaml`, `iptables rules.v4`) reflects the fix.
2. The service is restarted cleanly (not just signalled with `kill -HUP`).
3. The binding or state is re-verified **after the restart**.

**Why:** fixing a running process (editing an env var and running `export`) fixes the current session. The config file is what the next restart reads. If you only fix the process, the next reboot undoes your fix.

Always check the file, not just `ss -tlnp`.

## Security Checklist

Run monthly or after any infrastructure change.

```bash
# 1. Are there unexpected listening ports?
ss -tlnp

# 2. Do all UFW rules have active listeners?
ufw status numbered

# 3. Are fail2ban jails active?
fail2ban-client status

# 4. Are there unauthorized SSH keys?
cat ~/.ssh/authorized_keys

# 5. Are .env files permission-locked?
find /home /root /etc -name '*.env' -exec ls -la {} \;

# 6. Any processes not under systemd?
ps aux | grep -v '\['

# 7. Packages up to date?
apt list --upgradable 2>/dev/null | head -20

# 8. Docker containers all expected?
docker ps -a
```

## Service Ownership Rule

Every long-running process must be owned by systemd. No orphan processes started manually in a tmux session.

If a service needs to persist across reboots, it needs a unit file.

```ini
# /etc/systemd/system/my-service.service
[Unit]
Description=My Service
After=network.target

[Service]
Type=simple
User=root
WorkingDirectory=/opt/my-service
EnvironmentFile=/etc/my-service.env
ExecStart=/opt/my-service/run.sh
Restart=on-failure
RestartSec=5s

[Install]
WantedBy=multi-user.target
```

```bash
systemctl daemon-reload
systemctl enable my-service
systemctl start my-service
systemctl status my-service
```